# Lesson 16.5: Trace instruction identity, time and assumed cost

Read the matching lesson at `modules/16-harness-and-evidence/05-source-walkthrough.md`. Run this notebook from top to bottom in the project `.venv` kernel. All cases are fictional. The default path sends no model-provider requests. A kernel retains values between cells; restart and run all after an experiment to check independence.

## The practical question

A model result without its inputs, configuration and stopping reason is difficult to investigate. Harbour needs enough execution evidence to diagnose a failed contribution while avoiding a second uncontrolled store of customer information.

In [ ]:
from ai_orgdesign.labs import m16 as lab
print("Module 16 code loaded; no provider called.")

## Follow the mechanism

harness wraps the bounded loop from Module 9. It hashes a fixed instruction string, measures elapsed time with perf_counter, counts recorded tool calls and preserves the stop reason. It explicitly leaves provider_usage as None because the scripted planner makes no model request. estimate_cost multiplies token counts by assumed prices per million tokens and rejects negative inputs. The resulting number is illustrative arithmetic, not billed cost.

In [ ]:
import inspect
print(inspect.getsource(lab.estimate_cost))

In [ ]:
import inspect
print(inspect.getsource(lab.harness))

## Predict and run

The supplied loop uses one tool call and stops with final. Elapsed time is measured at runtime and will vary. The assumed cost calculation for one thousand input tokens at one unit per million and two hundred output tokens at four units per million is 0.0018 currency units. No actual provider usage is claimed.

In [ ]:
from ai_orgdesign.labs.m16 import demo
result = demo()
assert result["trace"]["provider_usage"] is None
assert abs(result["illustrative_cost"] - .0018) < 1e-12
print(result)

## Exercise

Suppose a run stopped after reading the case but before writing its final response. Decide which trace fields distinguish that failure from a denied tool call and from an unresolved business investigation.

Pause and write the predicted result and its interpretation before opening the worked answer below.

## Worked answer

A step-level record should distinguish requested, authorised, started, completed and failed operations, with bounded identifiers and exception categories. A final loop stop reason is not a full audit trail. An unresolved business case can coexist with a technically successful run; an execution error can coexist with a correct earlier read.

In [ ]:
from ai_orgdesign.labs.m16 import harness
from ai_orgdesign.labs.m09 import scripted_planner
print(harness(scripted_planner))

## Alternatives and organisational meaning

Structured local logs are sufficient for the first experiment. A tracing service can correlate distributed spans and evaluations, but needs retention, access and redaction rules. Recording hidden reasoning is neither required nor an adequate explanation of evidence support.

Explain separately what executed, what its output establishes and what remains a question for the surrounding service.